# 18 · Phase 14 — Final evaluation on the patient-independent CirCor test split

* frozen models, thresholds chosen on validation, **test split touched once here**
* recording-level metrics (primary: murmur present vs absent) and **patient-level** metrics
  (patient positive if any location is positive → max-probability aggregation)
* macro-F1, sensitivity, specificity, balanced accuracy, AUROC, AUPRC (mean ± std over 3 seeds),
  parameter count and inference time per recording
* bootstrap 95 % CI of the AUROC difference *proposed − best baseline* (patients resampled)
* *Unknown*-murmur recordings (excluded from training) reported separately
* secondary analysis: how well the murmur score ranks patients by clinical **outcome**

In [ ]:
%run common/00_config.ipynb
%run common/01_signal_lib.ipynb
%run common/02_rl_lib.ipynb
%run common/03_deep_lib.ipynb
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_auc_score, roc_curve

meta = pd.read_csv(P["processed"] / "proxy_records.csv")
proposed = load_json(P["results"] / "proposed_model.json")["proposed_model"]
MODELS = {"B1_raw_cnn": "Raw-waveform CNN", "B2_stft_cnn": "Fixed STFT CNN", "B3_db6_wavelet_cnn": "Fixed db6 wavelet CNN",
          "B4_fixed_TS_fusion": "Fixed temporal–spectral fusion", "B5_fixed_TSW_fusion": "Fixed tri-branch fusion (no RL)",
          "B6_supernet_ft_fixed_config": "Supernet fine-tuned, fixed config (no RL)",
          "R1_rl_wavelet_only": "RL wavelet-only", "R2_full_rl_frozen_policy": "Full RL fusion (frozen policy)",
          "R3_full_rl_finetuned": "Full RL fusion, supernet fine-tuned (frozen policy)", "R4_full_rl_joint_refined": "Full RL fusion + joint refinement"}
print("proposed model (chosen on validation):", MODELS[proposed])


def preds(name, seed):
    return pd.read_csv(P["models"] / name / f"preds_s{seed}.csv").merge(meta[["record_id", "patient_id"]], on="record_id")


rows, prow = [], []
for name, label in MODELS.items():
    for seed in CFG["deep"]["seeds"]:
        sm = load_summary(name, seed, P)
        d = preds(name, seed)
        te = d[d.split == "test"]
        m = binary_metrics(te.y, te.prob, sm["threshold"])
        rows.append({"model": label, "seed": seed, **m, "params": sm["params"], "infer_ms": sm["infer_ms_per_recording"]})
        yp, pp = patient_level(te.assign(patient_y=te.y), "prob", "patient_y")
        prow.append({"model": label, "seed": seed, **binary_metrics(yp, pp, sm["threshold"])})
rec = pd.DataFrame(rows)
pat = pd.DataFrame(prow)
main = mean_std_table(rec.to_dict("records"))
main["params"] = rec.groupby("model", sort=False).params.first().map(lambda v: f"{v / 1e6:.2f} M")
main["inference (ms/recording)"] = rec.groupby("model", sort=False).infer_ms.mean().round(1)
main.to_csv(P["tables"] / "18_main_results_recording_level.csv")
main

In [ ]:
pat_tab = mean_std_table(pat.to_dict("records"))
pat_tab.to_csv(P["tables"] / "18_main_results_patient_level.csv")
pat_tab

## Is the improvement real? Paired patient-bootstrap of the test AUROC difference

In [ ]:
base_names = [n for n in MODELS if n.startswith("B")]
best_base = max(base_names, key=lambda n: np.mean([load_summary(n, s, P)["val_auroc"] for s in CFG["deep"]["seeds"]]))
print("best baseline on validation:", MODELS[best_base])


def ens(name):
    ds = [preds(name, s) for s in CFG["deep"]["seeds"]]
    d = ds[0][["record_id", "patient_id", "split", "y"]].copy()
    d["prob"] = np.mean([x.prob.values for x in ds], 0)
    return d[d.split == "test"].reset_index(drop=True)


a, b = ens(proposed), ens(best_base)
assert (a.record_id == b.record_id).all()
rng = np.random.default_rng(0)
pids = a.patient_id.unique()
by_pid = {p: np.where(a.patient_id.values == p)[0] for p in pids}
diffs = []
for _ in range(2000):
    idx = np.concatenate([by_pid[p] for p in rng.choice(pids, len(pids))])
    if a.y.values[idx].min() == a.y.values[idx].max():
        continue
    diffs.append(roc_auc_score(a.y.values[idx], a.prob.values[idx]) - roc_auc_score(b.y.values[idx], b.prob.values[idx]))
diffs = np.array(diffs)
boot = {"proposed": MODELS[proposed], "baseline": MODELS[best_base],
        "auroc_proposed_seed_ensemble": roc_auc_score(a.y, a.prob), "auroc_baseline_seed_ensemble": roc_auc_score(b.y, b.prob),
        "delta_mean": float(diffs.mean()), "ci95": [float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))],
        "p_delta_le_0": float((diffs <= 0).mean())}
save_json(boot, P["results"] / "bootstrap_auroc_difference.json")
boot

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for name in [best_base, "B1_raw_cnn", "B2_stft_cnn", "R1_rl_wavelet_only", proposed]:
    d = ens(name)
    f, t, _ = roc_curve(d.y, d.prob); ax[0].plot(f, t, label=f"{MODELS[name]} ({roc_auc_score(d.y, d.prob):.3f})")
    pr, rc, _ = precision_recall_curve(d.y, d.prob); ax[1].plot(rc, pr, label=MODELS[name])
ax[0].plot([0, 1], [0, 1], "k:"); ax[0].set_xlabel("1 − specificity"); ax[0].set_ylabel("sensitivity"); ax[0].legend(fontsize=8); ax[0].set_title("ROC (test, seed ensemble)")
ax[1].axhline(a.y.mean(), color="k", ls=":"); ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].set_title("Precision–recall (test)"); ax[1].legend(fontsize=8)
plt.tight_layout(); savefig(fig, "18_roc_pr"); plt.show()

## Unknown-murmur recordings and the secondary outcome task

In [ ]:
import torch
device = torch.device("cuda")
space = ConfigSpace(CFG)
qc = pd.read_csv(P["metadata"] / "records_qc.csv")
ctx = pd.read_csv(P["processed"] / "contexts.csv")
unk = qc[(qc.murmur_label == "Unknown") & qc.qc_keep & (qc.split == "test")].merge(ctx, on="record_id")
sig = np.load(P["segments"] / "signals.npz")
pol = load_json(P["results"] / "policies" / ("rl_full.json" if proposed != "R4_full_rl_joint_refined" else "rl_joint_refined_s0.json"))
m0 = load_model(proposed, 0, CFG, P).to(device)
st = RecordStore([sig[r].astype(np.float32) for r in unk.record_id], policy_configs(unk, space, np.array(pol["policy"]), pol["K"]),
                 CFG["preprocessing"]["fs"], CFG["deep"])
unk["prob"] = predict(m0, st, np.arange(len(unk)), device)
thr0 = load_summary(proposed, 0, P)["threshold"]
d0 = preds(proposed, 0)
d0 = d0[d0.split == "test"]
fig, ax = plt.subplots(figsize=(8, 3.8))
for lab_, g in [("Absent", d0[d0.y == 0].prob), ("Present", d0[d0.y == 1].prob), ("Unknown (not trained on)", unk.prob)]:
    ax.hist(g, bins=30, alpha=0.5, density=True, label=f"{lab_} (n={len(g)})")
ax.axvline(thr0, color="k", ls="--", label="validation threshold"); ax.legend(); ax.set_xlabel("predicted murmur probability")
plt.tight_layout(); savefig(fig, "18_unknown_murmur"); plt.show()
print(f"Unknown test recordings flagged as murmur: {(unk.prob >= thr0).mean():.1%} (n={len(unk)})")

In [ ]:
out_rows = []
test_pat = qc[qc.split == "test"].groupby("patient_id").outcome.first()
for name in MODELS:
    d = ens(name)
    pp = d.groupby("patient_id").prob.max()
    yo = (test_pat.loc[pp.index] == "Abnormal").astype(int)
    out_rows.append({"model": MODELS[name], "outcome AUROC (patient max murmur prob)": roc_auc_score(yo, pp)})
out_tab = pd.DataFrame(out_rows)
out_tab.to_csv(P["tables"] / "18_secondary_outcome.csv", index=False)
out_tab.round(3)